# 5 · Embed in your own website (`roadstyle.js`)

This is the decoupled path: **Python computes the styling once** (a JSON spec), and a small, reusable **`roadstyle.js`** draws it in the browser with your own UI on top. No Python at runtime, and you don't write the renderer — it ships inside the package.

We'll generate a self-contained `output/web/` folder you can open with a double-click.

In [ ]:
import pathlib
import geopandas as gpd
import roadstyle as rs

edges = gpd.read_file(pathlib.Path("data") / "sundbyberg_edges.gpkg")
print(f"{len(edges):,} edges, CRS {edges.crs}")
edges[[c for c in edges.columns if c != edges.geometry.name]].head(3)


### 1 · Bake the spec + copy the renderer
`to_spec` precomputes every edge's style; `roadstyle.js` + `roadstyle.css` come straight from the installed package.

In [ ]:
import json, shutil, pathlib
from importlib.resources import files

web = pathlib.Path("output") / "web"
web.mkdir(parents=True, exist_ok=True)

spec = rs.to_spec(edges, basemap="dark_matter", palette="highsat")   # bake styling into a JSON spec
rs.save_spec(spec, web / "map_data.json")                   # also keep it as a file (served pattern)

for asset in ("roadstyle.js", "roadstyle.css"):             # the renderer, from the package
    shutil.copy(files("roadstyle") / "static" / asset, web / asset)

print("wrote", *(p.name for p in web.iterdir()))


### 2 · Tune interactions without touching code
`roadstyle.js` reads an optional interaction config — hover colour, selection glow, map toggles. A designer can edit this; the JS never changes.

In [ ]:
config = {
    "hoverColor": "#38bdf8",
    "hoverExtraWidth": 3,
    "selectionStyle": {
        "glow":   {"color": "#38bdf8", "width": 14, "opacity": 0.5},
        "casing": {"color": "#0f172a", "width": 8,  "opacity": 1.0},
        "core":   {"color": "#ffffff", "width": 4,  "opacity": 1.0}
    },
    "doubleClickZoom": True,
    "zoomControl": True,
}
(web / "interaction_config.json").write_text(json.dumps(config, indent=2))


### 3 · A page with a custom sidebar + a selected-road detail panel
The renderer is headless: it exposes `getRoadClasses()`, `setFilter()`, and a selection callback (`on('select', …)`). We **inline the spec** so the file opens with a double-click — browsers block `fetch()` from `file://`, so fetching `map_data.json` only works when the folder is *served* (`python -m http.server`). `load()` takes a URL or an object, so switching to the fetch pattern is a one-line change.

In [ ]:
PAGE = """<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8"><title>Road network</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<link rel="stylesheet" href="./roadstyle.css"/>
<style>
  body{margin:0;display:flex;height:100vh;font-family:system-ui,sans-serif;background:#0f172a;color:#e2e8f0}
  #sidebar{width:240px;padding:18px;box-sizing:border-box;border-right:1px solid #1e293b;overflow:auto}
  #sidebar h1{font-size:16px;margin:0 0 4px;color:#38bdf8}
  #sidebar p{font-size:12px;color:#94a3b8;margin:0 0 14px}
  label{display:flex;gap:8px;align-items:center;font-size:13px;padding:4px 6px;border-radius:6px}
  label:hover{background:#1e293b}
  #map{flex:1;height:100vh}
</style></head>
<body>
  <div id="sidebar">
    <h1>Road network</h1>
    <p>Styling precomputed in Python; filtered live in the browser.</p>
    <div id="filters"></div>
    <div id="detail" class="rs-detail">Click a road to see its details.</div>
  </div>
  <div id="map"></div>
  <script src="./roadstyle.js"></script>
  <script>
    // Spec + config are inlined so this page opens with a plain double-click (no server).
    // For a live app, serve the folder and fetch instead:
    //   m.load("./map_data.json", "./interaction_config.json")   // load() takes a URL or an object
    const SPEC = __SPEC__;
    const CONFIG = __CONFIG__;
    const m = new RoadStyleMap("map", { widgets: { legend: true } });
    m.load(SPEC, CONFIG).then(() => {
      const box = document.getElementById("filters");
      m.getRoadClasses().forEach(cls => {
        const l = document.createElement("label");
        l.innerHTML = `<input type="checkbox" value="${cls}" checked> ${cls}`;
        box.appendChild(l);
      });
      box.addEventListener("change", () => {
        const on = [...box.querySelectorAll("input:checked")].map(c => c.value);
        m.setFilter(on);
      });
      const detail = document.getElementById("detail");
      m.on("select", f => {
        const p = f.properties;
        detail.innerHTML = "<h4>Selected road</h4>" +
          Object.keys(p).filter(k => !k.startsWith("__rs_"))
            .map(k => `<div><b>${k}</b>: ${p[k] ?? ""}</div>`).join("");
      });
      m.on("deselect", () => { detail.innerHTML = "Click a road to see its details."; });
    });
  </script>
</body></html>
"""
PAGE = PAGE.replace("__SPEC__", json.dumps(spec)).replace("__CONFIG__", json.dumps(config))
(web / "index.html").write_text(PAGE)
print("Open", (web / "index.html").resolve(), "— double-click works, no server needed.")


### Zero-JavaScript alternative
If you don't need a custom UI, `save()` / `to_iframe()` give you a finished map with no frontend work (notebook 4). Reach for `roadstyle.js` only when you want your *own* page around the map.